# Embedding-Space OOD Detection

This notebook checks whether an observation is OOD by comparing its
learned embedding against the test set distribution.

Methods:
1. **PCA visualization** — project embeddings to 2D, visually inspect
2. **Mahalanobis distance** — parametric outlier score
3. **k-NN distance** — non-parametric outlier score

In [ ]:
import sys
from pathlib import Path
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))

from sbi4atmret.config.configs import BaseConfig
from sbi4atmret.models.ModelBase import BaseModel
from sbi4atmret.models.meta_learner import load_base_models
from sbi4atmret.evaluation.ood_tests import (
    analyze_embeddings,
    plot_embedding_pca,
    extract_embeddings,
    extract_obs_embedding,
    compute_pca,
    mahalanobis_distance,
    knn_distance,
)

## 1. Load Model and Data

In [ ]:
config_path = project_root / "experiments/config_MiriGeminiHST_cloudfree.yaml"

with open(config_path) as f:
    config_dict = yaml.safe_load(f)

config = BaseConfig(**config_dict)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load a single trained model
checkpoint_path = Path("path/to/states_800.pth")
models = load_base_models(
    [checkpoint_path],
    model_builder=lambda: BaseModel(config).build(),
    device=device,
)
model = models[0]
print(f"Model loaded, embedding output dim: check model.embedding")

In [ ]:
# Set up your test dataloader and batch_processor
# from sbi4atmret.datasets.DatasetBase import Dataset
# from sbi4atmret.runtime.batch_processor import BatchProcessor
#
# dataset = Dataset(config)
# test_loader = ...  # your test DataLoader
# batch_processor = BatchProcessor(pipe=..., noise=..., device=device)

# Load observation
# x_obs = torch.from_numpy(observation.full_observation).unsqueeze(0).float()

## 2. Full Analysis (One Call)

In [ ]:
result = analyze_embeddings(
    model,
    x_obs,
    dataloader=test_loader,
    batch_processor=batch_processor,
    n_components=2,
    k=5,
    device=device,
    max_batches=50,  # limit for speed
)

print(f"Test embeddings shape: {result.test_embeddings.shape}")
print(f"Obs embedding shape: {result.obs_embedding.shape}")
print(f"Mahalanobis distance: {result.mahalanobis_distance:.3f}")
print(f"k-NN distance: {result.knn_distance:.3f}")

## 3. PCA Visualization

In [ ]:
fig = plot_embedding_pca(result, title="Embedding PCA — Test Set vs Observation")
plt.show()

## 4. Higher-Dimensional PCA

In [ ]:
# Fit PCA with more components to check explained variance
from sklearn.decomposition import PCA

pca_full = PCA(n_components=min(20, result.test_embeddings.shape[1]))
pca_full.fit(result.test_embeddings)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(pca_full.explained_variance_ratio_)),
       pca_full.explained_variance_ratio_, color="steelblue")
ax.set_xlabel("PC")
ax.set_ylabel("Explained variance ratio")
ax.set_title(f"Cumulative: {pca_full.explained_variance_ratio_[:5].sum():.1%} in first 5 PCs")
plt.tight_layout()
plt.show()

In [ ]:
# 3D PCA scatter
test_pca_3d, obs_pca_3d, _ = compute_pca(
    result.test_embeddings, result.obs_embedding, n_components=3
)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    test_pca_3d[:, 0], test_pca_3d[:, 1], test_pca_3d[:, 2],
    alpha=0.2, s=5, color="steelblue",
)
ax.scatter(
    obs_pca_3d[:, 0], obs_pca_3d[:, 1], obs_pca_3d[:, 2],
    s=200, color="red", marker="*",
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("3D Embedding PCA")
plt.tight_layout()
plt.show()

## 5. Compare Multiple Models

If you have multiple models, check whether the observation is OOD
in all embedding spaces or only some.

In [ ]:
# checkpoint_paths = [...]
# all_models = load_base_models(checkpoint_paths, model_builder=..., device=device)
#
# for i, m in enumerate(all_models):
#     r = analyze_embeddings(m, x_obs, test_loader, batch_processor, device=device)
#     print(f"Model {i+1}: Mahalanobis={r.mahalanobis_distance:.3f}, k-NN={r.knn_distance:.3f}")
#     fig = plot_embedding_pca(r, title=f"Model {i+1}")
#     plt.show()

## 6. Threshold Calibration

Compute distances for all test samples to establish a baseline distribution.

In [ ]:
# Leave-one-out Mahalanobis distances for the test set
from scipy.spatial.distance import mahalanobis as scipy_mahal

mean = result.test_embeddings.mean(axis=0)
cov = np.cov(result.test_embeddings.T) + 1e-6 * np.eye(result.test_embeddings.shape[1])
cov_inv = np.linalg.inv(cov)

test_mahal = np.array([
    scipy_mahal(e, mean, cov_inv)
    for e in result.test_embeddings[:500]  # subset for speed
])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(test_mahal, bins=40, alpha=0.7, color="steelblue", label="Test set")
ax.axvline(
    result.mahalanobis_distance, color="red", linewidth=2,
    linestyle="--", label=f"Obs ({result.mahalanobis_distance:.2f})",
)
ax.set_xlabel("Mahalanobis distance")
ax.set_ylabel("Count")
ax.set_title("Distance distribution: Test vs Observation")
ax.legend()
plt.tight_layout()
plt.show()

# Percentile of the observation
percentile = (test_mahal < result.mahalanobis_distance).mean() * 100
print(f"Observation is at the {percentile:.1f}th percentile of test distances")